# Simple LLM Workflow

### Read the 1_bmi_workflow to learn more about the concepts in action or the info md files.

In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [ ]:
# -----------------------------------------------------------
# STEP 1 — Initialize the LLM model
# -----------------------------------------------------------
# Here we create an instance of the Gemini chat model through
# the LangChain wrapper.
#
# model="gemini-2.5-flash"
#   → a fast and efficient Gemini model suitable for most
#     conversational or question-answering tasks.
#
# temperature=0.7
#   → controls randomness of the generated output.
#   → lower values (0.2–0.3) produce more deterministic answers
#   → higher values (0.7–1.0) produce more creative responses.
#
# This model object will later be used inside a LangGraph node
# to generate responses from the LLM.

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [ ]:
# -----------------------------------------------------------
# STEP 2 — Define the graph state structure
# -----------------------------------------------------------
# LangGraph workflows pass information between nodes using a
# shared state object (a dictionary).
#
# TypedDict allows us to define the expected structure and
# types of this state, which improves:
# - code readability
# - type checking
# - debugging
#
# Fields:
# question → the user query
# answer   → the response generated by the LLM

# Create a state
class LLMState(TypedDict):
    question: str
    answer: str

In [ ]:
# -----------------------------------------------------------
# STEP 3 — Define the node function that interacts with the LLM
# -----------------------------------------------------------
# In LangGraph, every node is simply a Python function that:
#
# 1. Receives the current state
# 2. Performs some computation
# 3. Updates the state
# 4. Returns the updated state
#
# This node performs a simple question-answering task using
# the Gemini model.

def llm_qa(state: LLMState) -> LLMState:
    # extract the question from state
    question = state['question']

    # form a prompt for the LLM
    prompt = f"Answer the following question: {question} "
    
    # ask that question to the LLM
    # model.invoke() sends the prompt to Gemini and returns
    # an AIMessage object containing the response.
    # .content extracts the text generated by the model.
    response = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = response

    return state

In [8]:
# Create a graph
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# define edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [ ]:
# execute
initial_state = {'question': "What is the capital of France?"}

final_state = workflow.invoke(initial_state)

print(final_state['answer'])


The capital of France is **Paris**.


In [10]:
# We could have just done
# model.invoke("Answer the following question: What is the capital of France?").content
# But this workflow structure allows us to understand the flow of langgraph.